# Data Exploration (Cleaned Data)
สำรวจข้อมูลจาก cleaned files ใน `cleaned/` folder:
- `master_results_cleaned.csv` — คะแนนผู้สมัคร/พรรค รายหน่วยเลือกตั้ง
- `master_summary_cleaned.csv` — สรุปบัตร (ทั้งหมด/ดี/เสีย/งดออกเสียง) รายหน่วย
- `coords_cleaned.csv` — ที่ตั้งหน่วยเลือกตั้งพร้อมพิกัดครบถ้วน

In [89]:
import pandas as pd
import numpy as np
from pathlib import Path

CLEAN = Path('cleaned/')

results = pd.read_csv(CLEAN / 'master_results_cleaned.csv')
summary = pd.read_csv(CLEAN / 'master_summary_cleaned.csv')
coords  = pd.read_csv(CLEAN / 'coords_cleaned.csv')

# แยก ในเขต / นอกเขต
r_inzone  = results[results['unit_number'] != -1].copy()
r_outside = results[results['unit_number'] == -1].copy()
s_inzone  = summary[summary['unit_number'] != -1].copy()
s_outside = summary[summary['unit_number'] == -1].copy()

print('Loaded cleaned data:')
for name, df in [('results', results), ('summary', summary), ('coords', coords)]:
    print(f'  {name:10s}: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'\nresults  ในเขต: {len(r_inzone):,}  |  นอกเขต: {len(r_outside):,}')
print(f'summary  ในเขต: {len(s_inzone):,}  |  นอกเขต: {len(s_outside):,}')

Loaded cleaned data:
  results   : 19,500 rows x 7 cols
  summary   : 707 rows x 10 cols
  coords    : 342 rows x 8 cols

results  ในเขต: 18,849  |  นอกเขต: 651
summary  ในเขต: 681  |  นอกเขต: 26


---
## 1. master_results_cleaned.csv

In [90]:
results.head(10)

,type,province,district,sub-district,unit_number,name,score
0,เขต,ลำปาง,งาว,นาแก,1,นางสาววิชุดา ว่องวัฒนวิโรจน์,2
1,เขต,ลำปาง,งาว,นาแก,1,นางสาวสุวิภา กุศลจูง,39
2,เขต,ลำปาง,งาว,นาแก,1,นายศรีพรหม หอมยก,7
3,เขต,ลำปาง,งาว,นาแก,1,นายสมบูรณ์ รูปสะอาด,0
4,เขต,ลำปาง,งาว,นาแก,1,นายดาซัย เอกปฐพี,143
5,เขต,ลำปาง,งาว,นาแก,1,นายธนาธร โล่ห์สุนทร,11
6,เขต,ลำปาง,งาว,นาแก,1,พันเอกสันทัด ภัทรกิตตินนท์,2
7,เขต,ลำปาง,งาว,นาแก,3,นางสาววิชุดา ว่องวัฒนวิโรจน์,0
8,เขต,ลำปาง,งาว,นาแก,3,นางสาวสุวิภา กุศลจูง,38
9,เขต,ลำปาง,งาว,นาแก,3,นายศรีพรหม หอมยก,1


In [91]:
print('dtypes:')
print(results.dtypes)
print('\nnull count:')
print(results.isnull().sum())

dtypes:
type            object
province        object
district        object
sub-district    object
unit_number      int64
name            object
score            int64
dtype: object

null count:
type            0
province        0
district        0
sub-district    0
unit_number     0
name            0
score           0
dtype: int64


In [92]:
print('unique type     :', results['type'].unique())
print('unique province :', results['province'].unique())
print('unique district :', results['district'].nunique(), 'อำเภอ')
print('unique sub-dist :', results['sub-district'].nunique(), 'ตำบล')
print('unique name     :', results['name'].nunique(), 'พรรค/ผู้สมัคร')

unique type     : ['เขต' 'บช']
unique province : ['ลำปาง']
unique district : 6 อำเภอ
unique sub-dist : 52 ตำบล
unique name     : 136 พรรค/ผู้สมัคร


In [93]:
results['score'].describe()

count    19500.000000
mean         9.724308
std         41.936370
min          0.000000
25%          0.000000
50%          1.000000
75%          4.000000
max       4000.000000
Name: score, dtype: float64

In [94]:
# คะแนนรวมแยก type (ในเขต)
r_inzone.groupby('type')['score'].sum()

type
บช     88665
เขต    83126
Name: score, dtype: int64

In [95]:
# Top พรรค (บช) และ Top ผู้สมัคร (เขต) แยกกัน
bch_inzone = r_inzone[r_inzone['type'] == 'บช']
ket_inzone = r_inzone[r_inzone['type'] == 'เขต']

top_party = bch_inzone.groupby('name')['score'].sum().sort_values(ascending=False).head(20)
print('Top 20 พรรค (บัญชีรายชื่อ, ในเขต):')
print(top_party.to_string())
print()

# Top 5 ผู้สมัครในแต่ละเขต (district)
print('Top 5 ผู้สมัคร แบ่งตามเขต:')
print('=' * 55)
for district, grp in ket_inzone.groupby('district'):
    top5 = grp.groupby('name')['score'].sum().sort_values(ascending=False).head(5)
    print(f'\n[{district}]')
    for rank, (name, score) in enumerate(top5.items(), 1):
        print(f'  {rank}. {name:<30s} {score:,}')


Top 20 พรรค (บัญชีรายชื่อ, ในเขต):
name
ประชาชน            23759
เพื่อไทย           20527
กล้าธรรม            7493
ภูมิใจไทย           5550
รวมไทยสร้างชาติ     5504
ประชาธิปัตย์        3631
เพื่อชาติไทย        3461
เศรษฐกิจ            2539
พลังไทยรักชาติ      1434
ใหม่                1030
รวมพลังประชาชน       987
ไทยทรัพย์ทวี         975
พลังประชารัฐ         726
เสรีรวมไทย           712
ประชาไทย             641
พลังเพื่อไทย         591
พลวัต                546
มิติใหม่             507
พลังธรรมใหม่         459
วิชชั่นใหม่          451

Top 5 ผู้สมัคร แบ่งตามเขต:

[งาว]
  1. นายดาซัย เอกปฐพี               12,930
  2. นางสาวสุวิภา กุศลจูง           5,493
  3. นายธนาธร โล่ห์สุนทร            3,316
  4. นางสาววิชุดา ว่องวัฒนวิโรจน์   629
  5. นายศรีพรหม หอมยก               388

[วังเหนือ]
  1. นายดาซัย เอกปฐพี               7,929
  2. นางสาวสุวิภา กุศลจูง           5,850
  3. นายธนาธร โล่ห์สุนทร            3,769
  4. นางสาววิชุดา ว่องวัฒนวิโรจน์   680
  5. นายศรีพรหม หอมยก               430


In [96]:
# คะแนนรวมแยกอำเภอ (ในเขต, บช)
r_inzone[r_inzone['type']=='บช'].groupby('district')['score'].sum().sort_values(ascending=False)

district
งาว           21692
วังเหนือ      21024
แจ้ห่ม        20303
เมืองปาน      17198
เมืองลำปาง     8448
Name: score, dtype: int64

---
## 2. master_summary_cleaned.csv

In [97]:
summary.head(10)

,type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots,remain_ballots
0,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,700.0,663.0,8.0,29.0,NaN
1,เขต,ลำปาง,นอกเขต,ชุดที่ 1,-1,700.0,605.0,32.0,63.0,NaN
2,บช,ลำปาง,นอกเขต,ชุดที่ 10,-1,700.0,668.0,8.0,24.0,NaN
3,เขต,ลำปาง,นอกเขต,ชุดที่ 10,-1,700.0,616.0,28.0,56.0,NaN
4,บช,ลำปาง,นอกเขต,ชุดที่ 11,-1,700.0,664.0,6.0,30.0,NaN
5,เขต,ลำปาง,นอกเขต,ชุดที่ 11,-1,700.0,607.0,28.0,65.0,NaN
6,บช,ลำปาง,นอกเขต,ชุดที่ 12,-1,700.0,666.0,7.0,27.0,NaN
7,เขต,ลำปาง,นอกเขต,ชุดที่ 12,-1,700.0,614.0,33.0,53.0,NaN
8,บช,ลำปาง,นอกเขต,ชุดที่ 13,-1,694.0,662.0,4.0,28.0,NaN
9,เขต,ลำปาง,นอกเขต,ชุดที่ 13,-1,694.0,599.0,37.0,58.0,NaN


In [98]:
print('dtypes:')
print(summary.dtypes)
print('\nnull count:')
print(summary.isnull().sum())

dtypes:
type                object
province            object
district            object
sub-district        object
unit_number          int64
total_ballots      float64
valid_ballots      float64
invalid_ballots    float64
no_vote_ballots    float64
remain_ballots     float64
dtype: object

null count:
type                0
province            0
district            0
sub-district        0
unit_number         0
total_ballots       0
valid_ballots       1
invalid_ballots     1
no_vote_ballots     1
remain_ballots     27
dtype: int64


In [99]:
ballot_cols = ['total_ballots','valid_ballots','invalid_ballots','no_vote_ballots']
summary[ballot_cols].describe()

,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots
count,707.000000,706.000000,706.000000,706.000000
mean,432.475248,280.794618,20.745042,12.957507
std,145.240761,114.460345,10.472260,10.737270
min,80.000000,49.000000,0.000000,0.000000
25%,330.000000,208.000000,13.000000,6.000000
50%,420.000000,266.000000,20.000000,11.000000
75%,520.000000,333.000000,27.000000,16.000000
max,760.000000,677.000000,66.000000,69.000000


In [100]:
# บัตรรวมแยก type (ในเขต)
s_inzone.groupby('type')[ballot_cols].sum()

,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots
type,,,,
บช,143820.0,91562.0,6759.0,3370.0
เขต,143840.0,90035.0,7447.0,4674.0


In [101]:
# % บัตรดี/เสีย/งดออกเสียง (ในเขต)
total = s_inzone[ballot_cols].sum()
pct = pd.DataFrame({
    'จำนวน': total,
    '%': (total / total['total_ballots'] * 100).round(2)
})
pct

,จำนวน,%
total_ballots,287660.0,100.00
valid_ballots,181597.0,63.13
invalid_ballots,14206.0,4.94
no_vote_ballots,8044.0,2.80


In [102]:
# บัตรรวมแยกอำเภอ (ในเขต)
s_inzone.groupby('district')[ballot_cols].sum().sort_values('total_ballots', ascending=False)

,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots
district,,,,
งาว,81240.0,49475.0,3376.0,1453.0
วังเหนือ,66980.0,43863.0,3307.0,1973.0
แจ้ห่ม,60440.0,39020.0,3100.0,2214.0
เมืองปาน,52880.0,33239.0,2865.0,1334.0
เมืองลำปาง,26120.0,16000.0,1558.0,1070.0


---
## 3. coords_cleaned.csv

In [103]:
coords.head(10)

,province,district,sub-district,unit_number,village,place,latitude,longitude
0,ลำปาง,งาว,นาแก,1,1,ศาลาเอนกประสงค์ บ้านทุ่งศาลา,18.784909,99.970754
1,ลำปาง,งาว,นาแก,2,2,ศาลาประชาคม วัดหนองเหียง,18.784909,99.970754
2,ลำปาง,งาว,นาแก,3,3,ศาลาวัดนาแก,18.784909,99.970754
3,ลำปาง,งาว,นาแก,4,4,ศาลากลางหมู่บ้าน บ้านแม่ฮ่าง,18.784909,99.970754
4,ลำปาง,งาว,นาแก,5,5,โรงเรียนแม่แป้นวิทยา,18.805735,99.918016
5,ลำปาง,งาว,นาแก,6,6,ศาลากลางหมู่บ้าน บ้านสันติสุข,18.784909,99.970754
6,ลำปาง,งาว,บ้านร้อง,1,1,ศาลากลางหมู่บ้าน บ้านนาแรม,18.883165,99.945183
7,ลำปาง,งาว,บ้านร้อง,2,2,ศาลากลางหมู่บ้าน บ้านร้อง,18.883165,99.945183
8,ลำปาง,งาว,บ้านร้อง,3,3,ศาลาวัดสบป๋อน,18.883165,99.945183
9,ลำปาง,งาว,บ้านร้อง,4,4,ศาลาประชาคมบ้านข่อย,18.883165,99.945183


In [104]:
print('dtypes:')
print(coords.dtypes)
print('\nnull count:')
print(coords.isnull().sum())

dtypes:
province         object
district         object
sub-district     object
unit_number       int64
village           int64
place            object
latitude        float64
longitude       float64
dtype: object

null count:
province        0
district        0
sub-district    0
unit_number     0
village         0
place           0
latitude        0
longitude       0
dtype: int64


In [105]:
has_coord = coords[['latitude','longitude']].notnull().all(axis=1).sum()
print(f'มีพิกัด: {has_coord}/{len(coords)} หน่วย ({has_coord/len(coords)*100:.1f}%)')
print('\nสถิติพิกัด:')
coords[['latitude','longitude']].describe()

มีพิกัด: 342/342 หน่วย (100.0%)

สถิติพิกัด:


,latitude,longitude
count,342.000000,342.000000
mean,18.806487,99.682376
std,0.219062,0.173967
min,18.388821,99.423898
25%,18.680245,99.557375
50%,18.765791,99.620915
75%,18.964975,99.915958
max,19.284192,100.041142


In [106]:
# จำนวนหน่วยต่ออำเภอ
coords.groupby('district')['unit_number'].count().sort_values(ascending=False)

district
งาว           90
วังเหนือ      84
แจ้ห่ม        72
เมืองปาน      65
เมืองลำปาง    31
Name: unit_number, dtype: int64

---
## 4. Joined Dataset (ในเขต)

In [107]:
JOIN = ['district', 'sub-district', 'unit_number']

# summary + coords
df = s_inzone.merge(
    coords[JOIN + ['village','place','latitude','longitude']],
    on=JOIN, how='left'
)

has_coord = df[['latitude','longitude']].notnull().all(axis=1).sum()
print(f'joined shape : {df.shape}')
print(f'มีพิกัด      : {has_coord}/{len(df)} ({has_coord/len(df)*100:.1f}%)')
df.head(5)

joined shape : (681, 14)
มีพิกัด      : 681/681 (100.0%)


,type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots,remain_ballots,village,place,latitude,longitude
0,เขต,ลำปาง,งาว,นาแก,1,300.0,204.0,8.0,2.0,86.0,1,ศาลาเอนกประสงค์ บ้านทุ่งศาลา,18.784909,99.970754
1,เขต,ลำปาง,งาว,นาแก,2,700.0,404.0,27.0,17.0,252.0,2,ศาลาประชาคม วัดหนองเหียง,18.784909,99.970754
2,เขต,ลำปาง,งาว,นาแก,3,360.0,219.0,14.0,11.0,116.0,3,ศาลาวัดนาแก,18.784909,99.970754
3,เขต,ลำปาง,งาว,นาแก,4,460.0,300.0,23.0,6.0,131.0,4,ศาลากลางหมู่บ้าน บ้านแม่ฮ่าง,18.784909,99.970754
4,เขต,ลำปาง,งาว,นาแก,5,640.0,437.0,29.0,15.0,159.0,5,โรงเรียนแม่แป้นวิทยา,18.805735,99.918016


In [108]:
# อำเภอที่ผู้เลือกตั้งมากที่สุด (valid_ballots)
df.groupby('district')['valid_ballots'].sum().sort_values(ascending=False)

district
งาว           49475.0
วังเหนือ      43863.0
แจ้ห่ม        39020.0
เมืองปาน      33239.0
เมืองลำปาง    16000.0
Name: valid_ballots, dtype: float64

In [109]:
# หน่วยที่มีบัตรเสียมากที่สุด Top 10
df[['district','sub-district','unit_number','place','invalid_ballots']]\
    .sort_values('invalid_ballots', ascending=False).head(10)

,district,sub-district,unit_number,place,invalid_ballots
676,เมืองลำปาง,บ้านเสด็จ,11,ศาลาอเนกประสงค์ประจำหมู่บ้าน,66.0
492,เมืองลำปาง,บ้านแลง,3,ศาลาอเนกประสงค์ประจำหมู่บ้าน,58.0
563,แจ้ห่ม,เมืองมาย,6,ศาลาอเนกประสงค์ประจำหมู่บ้าน,53.0
623,แจ้ห่ม,วิเชตนคร,2,ศาลาอเนกประสงค์ประจำหมู่บ้าน,52.0
531,แจ้ห่ม,ทุ่งผึ้ง,2,ศาลาอเนกประสงค์ประจำหมู่บ้าน,51.0
393,เมืองปาน,แจ้ซ้อน,11,ศาลาประชาคมบ้านแจ้ซ้อนเหนือ,50.0
460,เมืองปาน,เมืองปาน,1,อาคารอเนกประสงค์โรงเรียนบ้านทุ่งโป่ง,50.0
470,เมืองลำปาง,บ้านเสด็จ,1,ศาลาอเนกประสงค์บ้านทรายมูล 1,50.0
483,เมืองลำปาง,บ้านเสด็จ,5,วัดปงอ้อม,50.0
489,เมืองลำปาง,บ้านแลง,11,ศาลาอเนกประสงค์ประจำหมู่บ้าน,50.0


In [110]:
df.columns = df.columns.str.strip()
total_scores = df.groupby('name')['score'].sum().reset_index()
sorted_scores = total_scores.sort_values(by='score', ascending=False)
top_10 = sorted_scores.head(10)
print(top_10)

KeyError: 'name'